In [ ]:
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
import json
from collections import Counter
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import regex
import yaml
from lxml import etree
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor
from src.processor.log_procesor import LogProcessor
from src.utils import formatTimedelta  # loadViasFromTopos,
from src.utils import (
    dateFromText,
    getEstacionamientos,
    getFilesByDate,
    getFilesByWeek,
    getNumbers,
    guardarExcel,
    guardarExcelMulti,
    isEmpty,
    isValidCode,
    listTopos,
    loadEstaciones,
    localizeFecha,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    rellenarId,
    roundGroup,
    setEF,
    slidingWindow,
    sortElements,
    sortStrNumbers,
    splitDataframe,
    splitList,
    splitLongString,
    time2localtime,
)
from src.visualizacion.visualizaciones import (
    build_hierarchical_dataframe,
    sample_random_colors,
    setHoverInfo,
)
import seaborn as sns
import matplotlib.pyplot as plt
from src.utils.util import loadEstacionComercial
from src.api.APIs import getPlanificacionCirculacionesTecnicas
from datetime import datetime, timedelta

In [ ]:

from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from pathlib import Path
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.units import inch
from datetime import datetime
from reportlab.lib.colors import Color
from datetime import timedelta
from reportlab.platypus import PageBreak
# color24 = colors.qualitative.Dark24
# color12 = colors.qualitative.Set3

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    jCTC: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        jCTC=jCTC,
        pro=pro,
        maniobra= maniobra
    )
    historico = historico[
        (historico["Fecha"] >= pd.to_datetime(start_date))
        & (historico["Fecha"] <= pd.to_datetime(end_date))
    ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    # historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
    #     subset=["Movimiento"]
    # )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
from datetime import datetime, timedelta

start_date = "2026-01-06"
end_date = "2026-01-30"

start = datetime.strptime(start_date, "%Y-%m-%d").date()
end = datetime.strptime(end_date, "%Y-%m-%d").date()

# Diccionario con una lista por cada día (lunes=0, domingo=6)
days = {i: [] for i in range(7)}

current = start
while current <= end:
    days[current.weekday()].append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)

# Opcional: asignar nombres
week_days = {
    "lunes": days[0],
    "martes": days[1],
    "miércoles": days[2],
    "jueves": days[3],
    "viernes": days[4],
    "sábado": days[5],
    "domingo": days[6],
}

print(week_days)


In [ ]:


# Diccionario final: un DataFrame por día
historico_por_dia = {dia: pd.DataFrame() for dia in week_days}

for dia_semana, fechas in week_days.items():
    for fecha in fechas:
        siguiente = (
            datetime.strptime(fecha, "%Y-%m-%d") + timedelta(days=1)
        ).strftime("%Y-%m-%d")

        ntrenes = [rellenarId(el) for el in np.arange(100000)]

        historico = cargarHistorico(
            fecha,
            siguiente,
            [],
            ntrenes,
            xSIV=True,
            jCTC=False,
            pro=True,
            maniobra=True
        )

        historico = historico.sort_values(
            by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
        ).reset_index(drop=True)

        # Campos de fecha
        historico["Día"] = historico["Fecha"].dt.date
        historico["day_of_week"] = historico["Fecha"].dt.day_of_week
        historico["day_of_year"] = historico["Fecha"].dt.day_of_year
        historico["week_of_year"] = (historico["day_of_year"] / 7).astype(int)

        # Concatenar al DataFrame del día correspondiente
        historico_por_dia[dia_semana] = pd.concat(
            [historico_por_dia[dia_semana], historico],
            ignore_index=True
        )


In [ ]:
# import pandas as pd
# import numpy as np
# from datetime import datetime, timedelta
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # --- Fechas de ejemplo ---
# start_date = "2026-01-01"
# end_date = "2026-01-28"

# start = datetime.strptime(start_date, "%Y-%m-%d").date()
# end = datetime.strptime(end_date, "%Y-%m-%d").date()

# # Diccionario con una lista por cada día de la semana
# days = {i: [] for i in range(7)}
# current = start
# while current <= end:
#     days[current.weekday()].append(current.strftime("%Y-%m-%d"))
#     current += timedelta(days=1)

# week_days = {
#     "lunes": days[0],
#     "martes": days[1],
#     "miércoles": days[2],
#     "jueves": days[3],
#     "viernes": days[4],
#     "sábado": days[5],
#     "domingo": days[6],
# }

# # --- Lista global para registrar fechas problemáticas ---
# fechas_problema = []

# # --- Función que procesa todas las fechas de un mismo día de la semana ---
# def procesar_dia_semana(dia_semana, fechas, max_intentos=2):
#     df_total = pd.DataFrame()
#     for fecha in fechas:
#         intento = 0
#         exito = False
#         while intento < max_intentos and not exito:
#             try:
#                 siguiente = (
#                     datetime.strptime(fecha, "%Y-%m-%d") + timedelta(days=1)
#                 ).strftime("%Y-%m-%d")

#                 # Generar trenes
#                 ntrenes = [rellenarId(el) for el in np.arange(100000)]

#                 # Cargar histórico
#                 historico = cargarHistorico(
#                     fecha,
#                     siguiente,
#                     [],
#                     ntrenes,
#                     xSIV=True,
#                     jCTC=False,
#                     pro=True,
#                     maniobra=True
#                 )

#                 # Ordenar y añadir campos de fecha
#                 historico = historico.sort_values(
#                     by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
#                 ).reset_index(drop=True)
#                 historico["Día"] = historico["Fecha"].dt.date
#                 historico["day_of_week"] = historico["Fecha"].dt.day_of_week
#                 historico["day_of_year"] = historico["Fecha"].dt.day_of_year
#                 historico["week_of_year"] = (historico["day_of_year"] / 7).astype(int)

#                 df_total = pd.concat([df_total, historico], ignore_index=True)
#                 exito = True  # éxito al cargar
#             except Exception as e:
#                 intento += 1
#                 print(f"[{dia_semana}] Error al procesar {fecha}, intento {intento}: {e}")
#                 if intento == max_intentos:
#                     fechas_problema.append(fecha)
#     return dia_semana, df_total

# # --- Ejecutar en paralelo por día de la semana ---
# NUM_THREADS = 3  # cambiado de 7 a 3 hilos
# historico_por_dia = {}

# with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
#     futures = [executor.submit(procesar_dia_semana, dia, fechas) for dia, fechas in week_days.items()]
#     for future in as_completed(futures):
#         dia, df = future.result()
#         historico_por_dia[dia] = df

# # --- Notificar fechas problemáticas ---
# if fechas_problema:
#     print("Fechas con problemas tras 2 intentos:", fechas_problema)
# else:
#     print("Todos los datos cargados correctamente.")

# # --- Verificar resultados ---
# for dia, df in historico_por_dia.items():
#     print(f"{dia}: {len(df)} filas")


In [ ]:
# Prefijos de trenes no comerciales
prefijos_excluir = (
    "37", "38", "39", "4", "5", "6", "78", "79", "8", "9"
)

# Nuevo diccionario con solo trenes comerciales
comerciales_por_dia = {}

for dia, df in historico_por_dia.items():
    if df.empty:
        comerciales_por_dia[dia] = df
        continue

    df_comercial = df[
        ~df["NTécnico"].astype(str).str.startswith(prefijos_excluir)
    ].copy()

    df_comercial = df_comercial[
        ~df_comercial["Producto"].isin(["Material vacio RAM"])
    ].copy()

    comerciales_por_dia[dia] = df_comercial


In [ ]:
comerciales_por_dia["lunes"]

In [ ]:
### Eliminar las estaciones no comerciales

In [ ]:
estaciones = loadEstacionComercial()


In [ ]:
estaciones_comerciales = estaciones.loc[
    estaciones["Comercial"]
].copy()

In [ ]:
codigos_comerciales = estaciones_comerciales["Código"]

# Filtrar por estaciones comerciales, día a día
comerciales_final_por_dia = {}

for dia, df in comerciales_por_dia.items():
    if df.empty:
        comerciales_final_por_dia[dia] = df
        continue

    comerciales_final_por_dia[dia] = df[
        df["Código"].isin(codigos_comerciales)
    ].copy()

In [ ]:
comerciales_final_por_dia["martes"]

In [ ]:
# Movimientos válidos
movimientos_validos = ["LLEGADA", "SALIDA", "ORIGEN", "FIN"]

movimiento_por_dia = {}

for dia, df in comerciales_final_por_dia.items():
    if df.empty:
        movimiento_por_dia[dia] = df
        continue

    movimiento_por_dia[dia] = df[
        df["Movimiento"].isin(movimientos_validos)
    ].copy()


In [ ]:
movimiento_por_dia["lunes"]

In [ ]:
fname = Path(r"data/Vía Estacionamiento/Vía_estacionamiento.csv")

In [ ]:
via = pd.read_csv(fname,low_memory=False)

In [ ]:
via.rename(columns={"Fecha Origen Tren (YYYYMMDD) N": "FechaOrigen","Cod  PR":"Código","Desc  PR":"Nombre","Cod NUM_Tren":"NTécnico","Desc Delegacion/Gerencia PR":"Subdirección","Cod Producto":"Producto"}, inplace=True)

In [ ]:
via["FechaOrigen"].unique()

In [ ]:
via = via[~via["NTécnico"].isin(["Cod NUM_Tren"])].copy()

In [ ]:
via

In [ ]:
via["FechaOrigen_dt"] = pd.to_datetime(
    via["FechaOrigen"].astype(str),
    format="%Y%m%d"
)

In [ ]:
via["day_of_week"] = via["FechaOrigen_dt"].dt.day_of_week

In [ ]:
via["NTécnico"] = via["NTécnico"].astype(str)

In [ ]:
dias_semana = {
    0: "lunes",
    1: "martes",
    2: "miércoles",
    3: "jueves",
    4: "viernes",
    5: "sábado",
    6: "domingo",
}

via_por_dia = {}

for dow, nombre in dias_semana.items():
    df_dia = via[via["day_of_week"] == dow].copy()
    via_por_dia[nombre] = df_dia


In [ ]:
lunes_via = via_por_dia["martes"]

In [ ]:
lunes_movimiento = movimiento_por_dia["martes"]

In [ ]:
lunes_via[lunes_via["NTécnico"].isin(lunes_movimiento["NTécnico"])]

In [ ]:
via_filtrado_por_dia = {}

for dia in via_por_dia:
    print(dia)
    df_via = via_por_dia[dia]
    df_mov = movimiento_por_dia.get(dia)


    if df_via.empty or df_mov is None or df_mov.empty:
        print(f"No hay datos para el día {dia}")
        via_filtrado_por_dia[dia] = df_via.iloc[0:0].copy()
        continue

    via_filtrado_por_dia[dia] = df_via[
        df_via["NTécnico"].isin(df_mov["NTécnico"])
    ].copy()

In [ ]:
sub_df_por_dia = {}

for dia, df in via_filtrado_por_dia.items():
    if df.empty:
        sub_df_por_dia[dia] = []
        continue

    sub_df_por_dia[dia] = [
        group for _, group in df.groupby(["NTécnico", "Código"])
    ]


In [ ]:
filas_por_dia = {}

for dia, lista_sub_df in sub_df_por_dia.items():
    filas = []

    for sub_df in lista_sub_df:
        # comprobar valores únicos de Via Estacionamiento
        if sub_df["Via Estacionamiento"].nunique() == 1:
            filas.append({
                "día": dia,
                "NTécnico": sub_df["NTécnico"].iloc[0],
                "Código": sub_df["Código"].iloc[0],
                "Via Estacionamiento": sub_df["Via Estacionamiento"].iloc[0],
                "Subdirección": sub_df["Subdirección"].iloc[0],
                "Producto": sub_df["Producto"].iloc[0],
            })

    filas_por_dia[dia] = filas


In [ ]:
dataframes_por_dia = {}

for key, filas in filas_por_dia.items():
    dataframes_por_dia[key] = pd.DataFrame(filas)

In [ ]:
movimiento_unico = {}

for key in filas_por_dia.keys() & movimiento_por_dia.keys():
    df_via = pd.DataFrame(filas_por_dia[key])   # 👈 CLAVE
    df_mov = movimiento_por_dia[key]

    movimiento_unico[key] = df_mov[
        df_mov["NTécnico"].isin(df_via["NTécnico"])
    ].copy()

In [ ]:
movimiento_unico["martes"]

In [ ]:
sub_df_por_key = {}

for key, df in movimiento_unico.items():
    sub_df_por_key[key] = [
        group
        for _, group in df.groupby(["NTécnico", "FechaOrigen", "Código"])
    ]


In [ ]:
# sub_df_resultado_por_key = {}

# for key, grupos in sub_df_por_key.items():
#     sub_df_resultado_por_key[key] = []

#     for df in grupos:
#         df = df.drop_duplicates(subset=["Movimiento"], keep="last")
#         movimientos = set(df["Movimiento"])

#         if "ORIGEN" in movimientos:
#             df = df[df["Movimiento"] == "ORIGEN"]
#         elif "FIN" in movimientos or "SALIDA" in movimientos:
#             df = df[df["Movimiento"] == "LLEGADA"]

#         sub_df_resultado_por_key[key].append(df)


In [ ]:
sub_df_resultado_por_key = {}

for key, grupos in sub_df_por_key.items():
    filas_resultado = []  # acumulamos filas, no DataFrames

    for df in grupos:
        df = df.drop_duplicates(subset=["Movimiento"], keep="last")
        movimientos = set(df["Movimiento"])

        if "ORIGEN" in movimientos:
            df_filtrado = df[df["Movimiento"] == "ORIGEN"]
        elif "FIN" in movimientos or "SALIDA" in movimientos:
            df_filtrado = df[df["Movimiento"] == "LLEGADA"]
        else:
            continue

        # guardamos solo las filas 
        filas_resultado.extend(df_filtrado.to_dict("records"))

    # creamos UN DataFrame por key
    sub_df_resultado_por_key[key] = pd.DataFrame(filas_resultado)


In [ ]:
# df_resultado_por_key = {
#     key: pd.concat(lista_df, ignore_index=True)
#     for key, lista_df in sub_df_resultado_por_key.items()
#     if lista_df  # evita error si la lista está vacía
# }

In [ ]:
sub_df_2_por_key = {
    key: [
        group
        for _, group in df.groupby(["NTécnico", "Código"])
    ]
    for key, df in sub_df_resultado_por_key.items()
}

In [ ]:
if isinstance(sub_df_resultado_por_key, dict):
    df_resultado = pd.concat(
        sub_df_resultado_por_key.values(),
        ignore_index=True
    )
test = df_resultado.copy()
test["Fecha"] = pd.to_datetime(test["Fecha"]).dt.strftime("%Y-%m-%d")
df_pivot_por_key = {}

for dia, fechas in week_days.items():
    df_dia = test[test["Fecha"].isin(fechas)].copy()

    if df_dia.empty:
        df_pivot_por_key[dia] = pd.DataFrame()
        continue

    df_pivot_por_key[dia] = (
        df_dia.pivot_table(
            index=["NTécnico", "Código", "Nombre"],
            columns="Fecha",
            values="Vía",
            aggfunc="first"
        )
        .reset_index()
    )


In [ ]:
df_pivot_por_key["lunes"]

In [ ]:

#### Filtrar las filas fechas que no tiene mismo
mask_mismo_valor_por_key = {}

for dia, df_pivot in df_pivot_por_key.items():
    if df_pivot.empty:
        mask_mismo_valor_por_key[dia] = pd.Series(dtype=bool)
        continue

    cols_fechas = df_pivot.columns.difference(
        ["NTécnico", "Código", "Nombre"]
    )

    mask_mismo_valor_por_key[dia] = (
        df_pivot[cols_fechas]
        .apply(lambda x: x.dropna().nunique() <= 1, axis=1)
    )


In [ ]:
### Filtar las filas de fechas con algún valor nulo
mask_mismo_valor_por_key = {}

for dia, df_pivot in df_pivot_por_key.items():
    if df_pivot.empty:
        mask_mismo_valor_por_key[dia] = pd.Series(dtype=bool)
        continue

    cols_fechas = df_pivot.columns.difference(
        ["NTécnico", "Código", "Nombre"]
    )

    mask_mismo_valor_por_key[dia] = df_pivot[cols_fechas].notna().all(axis=1)



In [ ]:
df_validas_por_key = {}

for dia in df_pivot_por_key:
    df = df_pivot_por_key[dia]
    mask = mask_mismo_valor_por_key[dia]

    if df.empty:
        df_validas_por_key[dia] = df
    else:
        df_validas_por_key[dia] = df[mask].copy()


In [ ]:
df_final_por_key = {}
for key in filas_por_dia.keys() & df_validas_por_key.keys():
    df_via = pd.DataFrame(filas_por_dia[key])  # 👈 esto no siempre funciona si ya es lista de DataFrames
    df_validas = df_validas_por_key[key]
    if df_validas.empty:
        df_final_por_key[key] = df_validas
        continue
    df_final_por_key[key] = df_via.merge(
        df_validas,
        on=["NTécnico", "Código"],
        how="right"
    )

In [ ]:
df_final_por_key = {
    key: df.dropna(subset=["Via Estacionamiento"]).copy() if not df.empty else df
    for key, df in df_final_por_key.items()
}


In [ ]:
cols_fechas_por_key = {}

for key, df in df_final_por_key.items():
    if df.empty:
        cols_fechas_por_key[key] = []
        continue

    cols_fechas_por_key[key] = df.columns.difference(
        ["NTécnico", "Código", "Via Estacionamiento", "Subdirección", "Producto", "Nombre","día"]
    ).tolist()


In [ ]:
cols_fechas_por_key.keys()

In [ ]:
df_no_coincidentes_por_key = {}

for key, df in df_final_por_key.items():
    if df.empty or not cols_fechas_por_key[key]:
        df_no_coincidentes_por_key[key] = pd.DataFrame()
        continue

    cols_fechas = cols_fechas_por_key[key]

    mask = (
        df[cols_fechas].notna() &
        (df[cols_fechas] != df["Via Estacionamiento"].values[:, None])
    ).any(axis=1)

    df_no_coincidentes_por_key[key] = df[mask].copy()


In [ ]:
df_no_coincidentes_por_key["martes"]

In [ ]:
for key, df in df_no_coincidentes_por_key.items():
    if not df.empty:
        df["Subdirección"] = df["Subdirección"].str.replace(r"^RC", "SD", regex=True)


In [ ]:
Sin_brete_por_key = {}

for key, df in df_no_coincidentes_por_key.items():
    if df.empty or not cols_fechas_por_key[key]:
        Sin_brete_por_key[key] = pd.DataFrame()
        continue

    cols_fechas = cols_fechas_por_key[key]

    mask = df[cols_fechas].applymap(
        lambda x: str(x).isdigit() if pd.notna(x) else False
    ).all(axis=1)

    Sin_brete_por_key[key] = df[mask].copy()


In [ ]:
Sin_brete_por_key.keys()

In [ ]:
Sin_brete_por_key["martes"]

In [ ]:
Sin_brete_por_key_reordenado = {}

for key, df in Sin_brete_por_key.items():
    if df.empty or not cols_fechas_por_key[key]:
        Sin_brete_por_key_reordenado[key] = df
        continue

    cols_fechas = cols_fechas_por_key[key]

    Sin_brete_por_key_reordenado[key] = df[
        ["NTécnico", "Código", "Nombre", "Subdirección", "Producto", "Via Estacionamiento",] 
        + cols_fechas
    ].copy()


In [ ]:
for key, df in Sin_brete_por_key_reordenado.items():
    if not df.empty:
        df.rename(columns={"Via Estacionamiento": "Vía Planificada"}, inplace=True)


In [ ]:
Sin_brete_por_key_reordenado["martes"]

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
import os
from collections import Counter
import re

def crear_excel_consolidado(diccionario_dfs, nombre_archivo='planificacion_semanal.xlsx'):
    """
    Crea un Excel con 3 hojas según el producto (L, R, C), consolidando filas con 
    mismo NTécnico, Código, Producto y Vía Planificada.
    
    Args:
        diccionario_dfs: Dict con claves 'lunes' a 'domingo' y DataFrames como valores
        nombre_archivo: Nombre del archivo Excel a crear
    """
    
    # Definir el directorio de salida
    directorio_salida = r'C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual'
    
    # Crear el directorio si no existe
    os.makedirs(directorio_salida, exist_ok=True)
    
    # Ruta completa del archivo
    ruta_completa = os.path.join(directorio_salida, nombre_archivo)
    
    # Orden de los días
    dias_orden = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']
    
    # Productos a procesar
    productos = ['L', 'R', 'C']
    
    # Diccionario para almacenar los DataFrames consolidados por producto
    dfs_por_producto = {}
    
    for producto in productos:
        # Identificar todas las fechas organizadas por día
        fecha_a_dia = {}
        fechas_por_dia = {dia: [] for dia in dias_orden}
        
        for dia in dias_orden:
            if dia in diccionario_dfs:
                df = diccionario_dfs[dia]
                
                # Filtrar por producto
                df_producto = df[df['Producto'] == producto].copy() if 'Producto' in df.columns else pd.DataFrame()
                
                if df_producto.empty:
                    continue
                
                # Obtener columnas de fechas (las que parecen fechas)
                columnas_fechas = [col for col in df_producto.columns 
                                 if isinstance(col, str) and (col.startswith('2026-') or '-' in col)]
                
                for fecha in columnas_fechas:
                    if fecha not in fechas_por_dia[dia]:
                        fechas_por_dia[dia].append(fecha)
                        fecha_a_dia[fecha] = dia.capitalize()
        
        # Ordenar fechas dentro de cada día y crear lista ordenada por día de la semana
        todas_las_fechas_ordenadas = []
        for dia in dias_orden:
            if dia in diccionario_dfs and fechas_por_dia[dia]:
                # Ordenar las fechas de este día cronológicamente
                fechas_dia_ordenadas = sorted(fechas_por_dia[dia])
                todas_las_fechas_ordenadas.extend(fechas_dia_ordenadas)
        
        # Columnas fijas de identificación
        columnas_id = ['NTécnico', 'Código', 'Nombre', 'Subdirección', 'Producto', 'Vía Planificada']
        
        # Crear un diccionario para consolidar los datos
        datos_consolidados = {}
        
        for dia in dias_orden:
            if dia not in diccionario_dfs:
                continue
                
            df = diccionario_dfs[dia]
            
            # Filtrar por producto
            df_producto = df[df['Producto'] == producto].copy() if 'Producto' in df.columns else pd.DataFrame()
            
            if df_producto.empty:
                continue
            
            # Identificar columnas de fechas en este DataFrame
            columnas_fechas_df = [col for col in df_producto.columns 
                                 if isinstance(col, str) and (col.startswith('2026-') or '-' in col)]
            
            for idx, row in df_producto.iterrows():
                # Crear clave única para identificar registros duplicados
                clave = (
                    row.get('NTécnico', ''),
                    row.get('Código', ''),
                    row.get('Producto', ''),
                    row.get('Vía Planificada', '')
                )
                
                # Si es la primera vez que vemos esta combinación
                if clave not in datos_consolidados:
                    datos_consolidados[clave] = {
                        'NTécnico': row.get('NTécnico', ''),
                        'Código': row.get('Código', ''),
                        'Nombre': row.get('Nombre', ''),
                        'Subdirección': row.get('Subdirección', ''),
                        'Producto': row.get('Producto', ''),
                        'Vía Planificada': row.get('Vía Planificada', ''),
                    }
                    # Inicializar todas las fechas vacías
                    for fecha in todas_las_fechas_ordenadas:
                        datos_consolidados[clave][fecha] = ''
                
                # Agregar los valores de las fechas de este día
                for fecha in columnas_fechas_df:
                    if fecha in todas_las_fechas_ordenadas:
                        valor = row.get(fecha, '')
                        # Si ya hay un valor, concatenar
                        if datos_consolidados[clave][fecha] and valor:
                            datos_consolidados[clave][fecha] = f"{datos_consolidados[clave][fecha]}, {valor}"
                        elif valor:
                            datos_consolidados[clave][fecha] = valor
        
        # Convertir el diccionario consolidado a DataFrame
        if datos_consolidados:
            df_consolidado = pd.DataFrame(list(datos_consolidados.values()))
            
            # Reorganizar columnas: primero las fijas, luego las fechas ordenadas por día
            columnas_ordenadas = columnas_id + todas_las_fechas_ordenadas
            df_consolidado = df_consolidado[columnas_ordenadas]
            
            dfs_por_producto[producto] = {
                'df': df_consolidado,
                'fecha_a_dia': fecha_a_dia,
                'todas_fechas': todas_las_fechas_ordenadas,
                'columnas_id': columnas_id
            }
    
    # Crear el archivo Excel con múltiples hojas
    with pd.ExcelWriter(ruta_completa, engine='openpyxl') as writer:
        for producto in productos:
            if producto in dfs_por_producto:
                dfs_por_producto[producto]['df'].to_excel(
                    writer, 
                    sheet_name=producto, 
                    index=False
                )
    
    # Cargar el workbook para aplicar formato
    wb = load_workbook(ruta_completa)
    
    # Estilos
    fill_dias = {
        'Lunes': PatternFill(start_color='E6F3FF', end_color='E6F3FF', fill_type='solid'),
        'Martes': PatternFill(start_color='FFE6F0', end_color='FFE6F0', fill_type='solid'),
        'Miércoles': PatternFill(start_color='FFF4E6', end_color='FFF4E6', fill_type='solid'),
        'Jueves': PatternFill(start_color='E6FFE6', end_color='E6FFE6', fill_type='solid'),
        'Viernes': PatternFill(start_color='F0E6FF', end_color='F0E6FF', fill_type='solid'),
        'Sábado': PatternFill(start_color='FFFFE6', end_color='FFFFE6', fill_type='solid'),
        'Domingo': PatternFill(start_color='FFE6E6', end_color='FFE6E6', fill_type='solid')
    }
    
    font_bold = Font(bold=True, size=11)
    font_rojo = Font(color='FF0000', size=11)  # Fuente roja
    alignment_center = Alignment(horizontal='center', vertical='center')
    
    # Función para extraer todos los números de un texto
    def extraer_numeros(texto):
        """Extrae todos los números de un texto"""
        if not texto or pd.isna(texto):
            return []
        # Buscar todos los números (enteros y decimales)
        numeros = re.findall(r'\d+\.?\d*', str(texto))
        return numeros
    
    # Aplicar formato a cada hoja
    for producto in productos:
        if producto not in dfs_por_producto:
            continue
            
        ws = wb[producto]
        info = dfs_por_producto[producto]
        fecha_a_dia = info['fecha_a_dia']
        todas_las_fechas_ordenadas = info['todas_fechas']
        columnas_id = info['columnas_id']
        df = info['df']
        
        # Insertar fila en la parte superior para los días de la semana
        ws.insert_rows(1)
        
        # Encontrar las columnas de fechas y agrupar por día
        col_inicio_fechas = len(columnas_id) + 1
        
        # Agrupar fechas consecutivas por día
        grupos_dias = []
        dia_actual = None
        col_inicio = None
        
        for idx, fecha in enumerate(todas_las_fechas_ordenadas, start=col_inicio_fechas):
            dia_fecha = fecha_a_dia.get(fecha)
            
            if dia_fecha != dia_actual:
                if dia_actual is not None:
                    grupos_dias.append((dia_actual, col_inicio, idx - 1))
                dia_actual = dia_fecha
                col_inicio = idx
        
        # Agregar el último grupo
        if dia_actual is not None:
            grupos_dias.append((dia_actual, col_inicio, col_inicio_fechas + len(todas_las_fechas_ordenadas) - 1))
        
        # Combinar celdas y aplicar formato
        for dia, col_ini, col_fin in grupos_dias:
            if col_ini == col_fin:
                # Una sola columna
                celda = ws.cell(row=1, column=col_ini)
                celda.value = dia
            else:
                # Múltiples columnas - combinar
                letra_ini = get_column_letter(col_ini)
                letra_fin = get_column_letter(col_fin)
                ws.merge_cells(f'{letra_ini}1:{letra_fin}1')
                celda = ws.cell(row=1, column=col_ini)
                celda.value = dia
            
            # Aplicar formato
            celda.font = font_bold
            celda.alignment = alignment_center
            if dia in fill_dias:
                celda.fill = fill_dias[dia]
                
                # Aplicar el mismo color a todas las columnas de ese día
                for col in range(col_ini, col_fin + 1):
                    for row in range(2, ws.max_row + 1):
                        ws.cell(row=row, column=col).fill = fill_dias[dia]
        
        # Aplicar formato a los encabezados de columnas fijas (fila 2)
        for col in range(1, len(columnas_id) + 1):
            celda = ws.cell(row=2, column=col)
            celda.font = font_bold
            celda.fill = PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid')
            celda.alignment = alignment_center
        
        # Marcar en rojo el texto de las celdas cuyos números no coinciden con el mayoritario
        for row_idx, row_data in df.iterrows():
            # Obtener todos los números de las columnas de fechas
            numeros_por_celda = {}
            todos_numeros = []
            
            for fecha in todas_las_fechas_ordenadas:
                valor = row_data[fecha]
                numeros = extraer_numeros(valor)
                numeros_por_celda[fecha] = numeros
                
                # Agregar todos los números encontrados (pueden ser múltiples por celda)
                todos_numeros.extend(numeros)
            
            # Si hay números, encontrar el número mayoritario
            if todos_numeros:
                contador = Counter(todos_numeros)
                numero_mayoritario = contador.most_common(1)[0][0]
                
                # Revisar cada celda de fecha y marcar texto en rojo si sus números no coinciden
                for col_idx, fecha in enumerate(todas_las_fechas_ordenadas, start=col_inicio_fechas):
                    numeros_celda = numeros_por_celda.get(fecha, [])
                    
                    # La fila en Excel es row_idx + 3 (1 para día, 1 para encabezado, 1 porque pandas indexa desde 0)
                    excel_row = row_idx + 3
                    
                    celda = ws.cell(row=excel_row, column=col_idx)
                    
                    # Si la celda tiene números y ninguno coincide con el mayoritario, marcar en rojo
                    if numeros_celda and numero_mayoritario not in numeros_celda:
                        celda.font = font_rojo
        
        # Ajustar ancho de columnas
        for column in ws.columns:
            max_length = 0
            column_letter = get_column_letter(column[0].column)
            for cell in column:
                try:
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            adjusted_width = min(max_length + 2, 30)
            ws.column_dimensions[column_letter].width = adjusted_width
        
        # Congelar paneles (primera fila y columnas fijas)
        ws.freeze_panes = ws.cell(row=3, column=len(columnas_id) + 1)
    
    # Guardar
    wb.save(ruta_completa)
    
    # Mostrar estadísticas
    print(f"✓ Archivo guardado exitosamente en:")
    print(f"  {ruta_completa}")
    print(f"\nEstadísticas por producto:")
    for producto in productos:
        if producto in dfs_por_producto:
            df = dfs_por_producto[producto]['df']
            print(f"  - Hoja '{producto}': {len(df)} filas, {len(dfs_por_producto[producto]['todas_fechas'])} fechas")
        else:
            print(f"  - Hoja '{producto}': Sin datos")

# Uso
crear_excel_consolidado(Sin_brete_por_key_reordenado, 'fiabilidad_via_semanal.xlsx')

In [ ]:
# L_por_key = {}
# C_por_key = {}
# R_por_key = {}

# for key, df in Sin_brete_por_key_reordenado.items():
#     if df.empty:
#         L_por_key[key] = pd.DataFrame()
#         C_por_key[key] = pd.DataFrame()
#         R_por_key[key] = pd.DataFrame()
#         continue

#     L_por_key[key] = df[df["Producto"] == "L"].drop(columns=["Producto"]).copy()
#     C_por_key[key] = df[df["Producto"] == "C"].drop(columns=["Producto"]).copy()
#     R_por_key[key] = df[df["Producto"] == "R"].drop(columns=["Producto"]).copy()


In [ ]:
reemplazos = {
    "R": "Media distancia",
    "L": "Larga distancia",
    "C": "Cercanía"
}
for dia, datagrama in Sin_brete_por_key_reordenado.items():
    datagrama["Producto"] = datagrama["Producto"].astype(str)
    datagrama["Producto"] = datagrama["Producto"].replace(reemplazos)


In [ ]:
# data = {
#     "L": L,
#     "C": C,
#     "R": R
# }
orden_dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

# Crear un diccionario ordenado
data = {key: Sin_brete_por_key_reordenado[key] for key in orden_dias}

fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\2026-01-05-2026-01-30_fiabilidad_via.xlsx")

guardarExcelMulti(data, fname)

In [ ]:
historico_total[(historico_total["NTécnico"] == "08072") & (historico_total["Código"]== "92102")]